# Sentiment Analysis Pipeline - Optimized for Windows
## Big Data System: Hadoop + Spark

**Note**: This notebook has been optimized to run on resource-constrained hardware (8GB RAM).

## Stage 1: Setup Spark Session

In [9]:
import os

os.environ["SPARK_HOME"] = "/opt/conda"
os.environ["PATH"] += ":/opt/conda/bin"

In [11]:
from pyspark import SparkContext

try:
    SparkContext.getOrCreate().stop()
except:
    pass

In [13]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf() \
    .setAppName('Sentiment-Analysis-Notebook') \
    .setMaster('local[*]') \
    .set('spark.executor.memory', '1g') \
    .set('spark.driver.memory', '512m')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

print("Spark OK:", spark.version)

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## Stage 2: Load Data from HDFS

In [ ]:
schema = 'polarity INT, id STRING, date STRING, query STRING, user STRING, text STRING'
hdfs_path = 'hdfs://namenode:9000/sentiment/raw/sentiment140_full.csv'

print("Loading data from HDFS...")
df = spark.read.csv(hdfs_path, schema=schema, header=False)

# Show first 5 rows
df.select("polarity", "user", "text").show(5, truncate=50)

## Stage 3: Preprocessing (Cleaning)

In [ ]:
import re
from pyspark.sql.functions import col, lower, udf
from pyspark.sql.types import StringType

def clean_text(text):
    if not text: return ""
    text = re.sub(r'http\S+', '', str(text)) # Remove URLs
    text = re.sub(r'@\w+', '', text)        # Remove Mentions
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)# Remove special chars
    return " ".join(text.lower().split())

clean_udf = udf(clean_text, StringType())

# Take a sample for interactive analysis to save memory
sample_df = df.limit(50000).withColumn("cleaned_text", clean_udf(col("text")))

sample_df.select("text", "cleaned_text").show(5, truncate=60)

## Stage 4: Feature Extraction (TF-IDF)

In [ ]:
from pyspark.ml.feature import HashingTF, IDF, Tokenizer
from pyspark.ml import Pipeline

tokenizer = Tokenizer(inputCol="cleaned_text", outputCol="words")
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=10000)
idf = IDF(inputCol="rawFeatures", outputCol="features")

pipeline = Pipeline(stages=[tokenizer, hashingTF, idf])
pipeline_model = pipeline.fit(sample_df)
featurized_df = pipeline_model.transform(sample_df)

featurized_df.select("features").show(3, truncate=80)

## Stage 5: Machine Learning (Logistic Regression)

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Prepare labels (0 for negative, 1 for positive)
labeled_df = featurized_df.withColumn("label", (col("polarity") == 4).cast("double"))

# Train/Test Split
train_data, test_data = labeled_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training model on {train_data.count()} rows...")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
lr_model = lr.fit(train_data)

# Make predictions
predictions = lr_model.transform(test_data)

# Evaluate
evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction")
auc = evaluator.evaluate(predictions)
print(f"Area Under ROC: {auc:.4f}")

## Stage 6: Word Count (MapReduce style)

In [ ]:
from pyspark.sql.functions import explode, split

word_counts = sample_df.select(explode(split(col("cleaned_text"), " ")).alias("word")) \
    .filter(col("word") != "") \
    .groupBy("word") \
    .count() \
    .orderBy(col("count").desc())

print("Top 20 most frequent words:")
word_counts.show(20)